# Re-embed the PubMed abstracts with the fine-tuned BioBERT

Rewrites the `dense` vector of every abstract in the Zilliz collection
`abstract_search` with the fine-tuned encoder that PathwayOracle uses for
queries (`yoyo458/BioBERT_GOUniProt_FineTuned`, pinned revision), so query and
documents share one vector space.

**What changes and what doesn't**

| field | after this notebook |
|---|---|
| `pmid` (primary key) | unchanged |
| `text` | unchanged -- each row is read and written back with the identical string |
| `sparse` (BM25) | regenerated by Zilliz from the same text -> identical |
| `dense` | **replaced** with the fine-tuned embedding (mean pooling, L2-normalised -- exactly as the server encodes queries) |

It is an in-place overwrite: the old vectors are not kept. While it runs, rows
already processed and rows not yet processed sit in different vector spaces, so
semantic search is mixed until it finishes.

**Before you start**

1. Runtime -> Change runtime type -> **GPU** (T4 is fine; L4/A100 faster).
2. Colab secrets (key icon, left sidebar): `ZILLIS_URI` and **`ZILLIS_WRITE_TOKEN`** -- a token
   that can *write* to the collection. The read-only token PathwayOracle serves with will be
   refused on upsert.
3. Optional: set `USE_DRIVE = True` so the checkpoint survives a disconnected runtime;
   re-running the main cell then resumes where it stopped.

**Rough time** for 563,036 abstracts at up to 512 tokens: T4 with fp16 ~25-40 min,
L4 ~15-25 min, A100 ~8-12 min -- plus whatever Zilliz's write rate allows. The
progress bar shows the real rate and ETA after the first few batches.

In [ ]:
!pip -q install -U "sentence-transformers>=5" "pymilvus>=2.6" huggingface_hub

## 1. Configuration

In [ ]:
import os

# --- what to rewrite ---------------------------------------------------------
COLLECTION = "abstract_search"
PK, TEXT, DENSE = "pmid", "text", "dense"

# --- the encoder: the same pinned weights the PathwayOracle server queries with -
MODEL = "yoyo458/BioBERT_GOUniProt_FineTuned"
REVISION = "dc05085095f1ae926aa014bb0a88d087bc0dce8f"          # Hub commit "Encoder 710f986575db"
WEIGHTS_SHA256 = "710f986575dbf663593f85214adaba5f3b7341551dc913c48389a23fe0d1f88b"

# --- throughput ------------------------------------------------------------------
READ_BATCH = 1000      # rows fetched from Zilliz per page
ENCODE_BATCH = 128     # sequences per GPU forward pass (lower to 64 on OOM)
UPSERT_BATCH = 500     # rows per upsert request
USE_FP16 = True        # ~2x faster on T4/L4; checked against fp32 in section 3
LOG_EVERY = 25         # also print a plain log line every N pages

# --- resumability ----------------------------------------------------------------
USE_DRIVE = False      # True: keep the checkpoint on Google Drive
CHECKPOINT_NAME = "reembed_abstracts_checkpoint.json"

# --- credentials (Colab secrets; never paste tokens into the notebook) -----------
try:
    from google.colab import userdata
    ZILLIS_URI = userdata.get("ZILLIS_URI")
    ZILLIS_TOKEN = userdata.get("ZILLIS_WRITE_TOKEN")
except Exception:                                    # not on Colab: fall back to env
    ZILLIS_URI = os.environ["ZILLIS_URI"]
    ZILLIS_TOKEN = os.environ["ZILLIS_WRITE_TOKEN"]

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT = f"/content/drive/MyDrive/{CHECKPOINT_NAME}"
else:
    CHECKPOINT = f"/content/{CHECKPOINT_NAME}"
print("checkpoint:", CHECKPOINT)

## 2. Connect and check the collection

In [ ]:
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri=ZILLIS_URI, token=ZILLIS_TOKEN, timeout=60)
desc = client.describe_collection(COLLECTION)
fields = {f["name"]: f for f in desc["fields"]}
functions = [(fn.get("name"), fn.get("input_field_names"), fn.get("output_field_names"))
             for fn in desc.get("functions", [])]

for f in desc["fields"]:
    print(f"  {f['name']:8} {f['type']!s:24} {f.get('params', {})}{'  PK' if f.get('is_primary') else ''}")
print("  functions:", functions)

assert fields[PK].get("is_primary"), "pmid is expected to be the primary key"
assert int(fields[DENSE]["params"]["dim"]) == 768, "dense must be 768-d for this encoder"
# The sparse field is generated by Zilliz (BM25 function over `text`), so it is
# never written by this notebook -- upserting the text regenerates it.
generated = {o for _, _, outs in functions for o in (outs or [])}
print("  generated by Zilliz, not written here:", sorted(generated))

TOTAL = client.get_collection_stats(COLLECTION)["row_count"]
print(f"\n{TOTAL:,} rows in {COLLECTION}")

## 3. Load the pinned encoder and verify it

In [ ]:
import hashlib, time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

# Verify the weights are exactly the ones PathwayOracle serves with.
path = Path(snapshot_download(MODEL, revision=REVISION))
h = hashlib.sha256()
with open(path / "model.safetensors", "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
assert h.hexdigest() == WEIGHTS_SHA256, f"weights sha256 {h.hexdigest()} != pinned {WEIGHTS_SHA256}"
print("weights sha256 OK")

model = SentenceTransformer(MODEL, revision=REVISION, device="cuda")
print("max_seq_length:", model.max_seq_length, "| pooling: mean, normalised")

# fp16 check: cosine between fp32 and fp16 embeddings of real abstracts.
sample = [r[TEXT] for r in client.query(COLLECTION, filter=f"{PK} > 0", output_fields=[TEXT], limit=64)]
fp32 = model.encode(sample, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
if USE_FP16:
    model.half()
    fp16 = model.encode(sample, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    cos = (fp32 * fp16.astype(np.float32)).sum(1)
    print(f"fp16 vs fp32 cosine: min {cos.min():.5f}, mean {cos.mean():.5f}")
    if cos.min() <= 0.999:
        # The stored vectors must match what the server computes in fp32 at query
        # time, so drift is not acceptable: fall back to fp32 instead of stopping.
        print("fp16 drifts beyond 0.999 cosine on real abstracts -> using fp32 (slower, exact)")
        USE_FP16 = False
        model = SentenceTransformer(MODEL, revision=REVISION, device="cuda")

# Throughput probe, so the ETA below is grounded in this GPU.
t = time.perf_counter()
model.encode(sample * 4, batch_size=ENCODE_BATCH, normalize_embeddings=True, show_progress_bar=False)
rate = len(sample) * 4 / (time.perf_counter() - t)
print(f"encode rate ~{rate:,.0f} abstracts/s -> ~{TOTAL / rate / 60:,.0f} min of GPU time for {TOTAL:,}")

## 4. Re-embed (resumable)

Reads pages in primary-key order, encodes on the GPU, and upserts `pmid` + the
**unchanged** `text` + the new `dense`, while the next page is being encoded.
After every page the last finished `pmid` goes to the checkpoint; re-running this
cell continues after it. Interrupt it (stop button) any time -- nothing is lost
beyond the page in flight.

In [ ]:
import json, logging, os, time
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S", force=True)
log = logging.getLogger("reembed")


def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            return json.load(f)
    return {"last_pk": 0, "done": 0, "started": time.time()}


def save_checkpoint(state):
    tmp = CHECKPOINT + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f)
    os.replace(tmp, CHECKPOINT)                      # atomic: never a half-written checkpoint


def upsert_with_retry(rows, attempts=6):
    for i in range(attempts):
        try:
            for k in range(0, len(rows), UPSERT_BATCH):
                client.upsert(COLLECTION, data=rows[k:k + UPSERT_BATCH])
            return
        except Exception as exc:                     # transient network / rate limits
            wait = min(60, 2 ** i)
            log.warning("upsert failed (%s); retry %d/%d in %ds", str(exc)[:120], i + 1, attempts, wait)
            time.sleep(wait)
    raise RuntimeError("upsert kept failing; re-run this cell to resume from the checkpoint")


def pages(after_pk):
    # Primary-key order + `pk > last` makes a restart pick up exactly where it stopped.
    it = client.query_iterator(COLLECTION, batch_size=READ_BATCH, filter=f"{PK} > {after_pk}",
                               output_fields=[PK, TEXT])
    try:
        while True:
            page = it.next()
            if not page:
                return
            yield page
    finally:
        it.close()


state = load_checkpoint()
log.info("starting after %s=%s (%s of %s already done)", PK, state["last_pk"], f"{state['done']:,}", f"{TOTAL:,}")
bar = tqdm(total=TOTAL, initial=state["done"], unit="abs", smoothing=0.05, dynamic_ncols=True)
pool = ThreadPoolExecutor(max_workers=1)             # upload page N while encoding page N+1
pending = None
t0, done0, n_pages = time.time(), state["done"], 0

try:
    for page in pages(state["last_pk"]):
        page.sort(key=lambda r: r[PK])
        texts = [r[TEXT] for r in page]
        vecs = model.encode(texts, batch_size=ENCODE_BATCH, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
        rows = [{PK: r[PK], TEXT: r[TEXT], DENSE: v.tolist()} for r, v in zip(page, vecs)]

        if pending is not None:                      # previous page's upload must land first
            pending.result()
            save_checkpoint(state)
        pending = pool.submit(upsert_with_retry, rows)
        state = {**state, "last_pk": page[-1][PK], "done": state["done"] + len(page)}

        n_pages += 1
        bar.update(len(page))
        rate = (state["done"] - done0) / max(time.time() - t0, 1e-6)
        bar.set_postfix(rate=f"{rate:,.0f}/s", last_pmid=state["last_pk"])
        if n_pages % LOG_EVERY == 0:
            eta_min = (TOTAL - state["done"]) / max(rate, 1e-6) / 60
            log.info("%s / %s (%.1f%%)  %.0f abs/s  ETA %.0f min  last %s=%s",
                     f"{state['done']:,}", f"{TOTAL:,}", 100 * state["done"] / TOTAL, rate, eta_min,
                     PK, state["last_pk"])
    if pending is not None:
        pending.result()
        save_checkpoint(state)
    log.info("finished: %s rows re-embedded in %.1f min", f"{state['done']:,}", (time.time() - t0) / 60)
finally:
    bar.close()
    pool.shutdown(wait=True)

## 5. Verify

In [ ]:
import random

# Row count unchanged.
after = client.get_collection_stats(COLLECTION)["row_count"]
print(f"rows: {after:,} (was {TOTAL:,})")

# A random sample: stored dense == the fine-tuned encoding of the stored text.
# PMIDs are sparse over a wide range, so sample random START points and take the
# next few rows after each, rather than guessing ids that may not exist.
lo, hi = 1, int(state["last_pk"])
rows = []
for start in random.sample(range(lo, hi), 10):
    rows += client.query(COLLECTION, filter=f"{PK} > {start}", output_fields=[PK, TEXT, DENSE], limit=5)
stored = np.array([r[DENSE] for r in rows], dtype=np.float32)
fresh = model.encode([r[TEXT] for r in rows], batch_size=32, normalize_embeddings=True,
                     convert_to_numpy=True).astype(np.float32)
cos = (stored * fresh).sum(1)
print(f"{len(rows)} sampled rows: cosine(stored, re-encoded) min {cos.min():.4f}, mean {cos.mean():.4f}")
assert cos.min() > 0.99, "some sampled rows still carry the old embedding"

In [ ]:
# Retrieval check: the query PathwayOracle got wrong before re-embedding.
from pymilvus import AnnSearchRequest, RRFRanker

query = "STAT3 represses p53 transcription"
vec = model.encode(query, normalize_embeddings=True).astype(np.float32).tolist()
hits = client.hybrid_search(
    COLLECTION,
    reqs=[AnnSearchRequest(data=["STAT3 p53"], anns_field="sparse", param={"drop_ratio_search": 0.0}, limit=100),
          AnnSearchRequest(data=[vec], anns_field=DENSE, param={"nprobe": 10}, limit=100)],
    ranker=RRFRanker(100), limit=5, output_fields=[PK, TEXT])[0]
for h in hits:
    print(h["entity"][PK], h["entity"][TEXT][7:100])
print("\nexpected near the top: 16107692 'Role of Stat3 in regulating p53 expression and function'")